In [ ]:
import numpy

In [ ]:
data_x_spectrum = numpy.load("x.npz")


In [ ]:
frequency = data_x_spectrum['frequency']
amplitude = data_x_spectrum['amplitude']

In [ ]:
import remove_edge_and_params as reap
import peak_finder_and_test_plot as pfat
import build_clusters_and_peak_spaces as bcps
import normalization_and_spectra as normspec
import crosscorrelation_and_template_matching as crosstemp
import remove_overrepresented_shapes as ros
import grouping
import first_harmonic as fh
import continuous_spectrum_and_analysis as csa

In [ ]:
frequency, amplitude = reap.remove_edge_artifacts(
    frequency,
    amplitude,
    mean_window=1001,
    gradient_window=501,
    gradient_threshold_factor=0.15,
    amplitude_threshold_factor=20,
    min_region=500,
    symmetric_edges=True,
    debug=True
)

In [ ]:
params = reap.estimate_spectrum_parameters(
    frequency,
    expected_frev=2.00e6,
    frev_tolerance=0.10e6
)

In [ ]:
# optional : Plot of the spectrum 

# import plotly.graph_objects
# fig = plotly.graph_objects.Figure()
# fig.add_trace(plotly.graph_objects.Scatter(x=frequency,
#     y=amplitude,
#     mode="lines",
#     name="Spectrum"
# ))
# fig.update_layout(
#     title="Interactive Spectrum",
#     xaxis_title="frequency",
#     yaxis_title="amplitude"
# )


# fig.show()


In [ ]:
peaks, resonance_mask, activity = pfat.find_peaks_adaptive(
        frequency,
        amplitude,
        prominence_resonance=0.003,   # these need to be adapted on the data (look in the spectrum an change to get all necesary peaks)
        distance_resonance=10,
        threshold_background=0.001,
        merge_background_points=7,
        activity_window=300,
        activity_fraction=0.40
    )

In [ ]:
# Plot in oder to see if the values are set correctly 
pfat.plot_spectrum_with_peaks_plotly(
    frequency,
    amplitude,
    peaks
)

In [ ]:
import numpy as np

In [ ]:
def run_pipeline_no_grouping( # not in package right now
    frequency,
    amplitude
):

    # peak detection

    peaks, resonance_mask, activity = pfat.get_adaptive_peaks(
        frequency,
        amplitude
    )

    # peak space 

    peak_freqs = frequency[peaks]

    peak_amps = amplitude[peaks]

    peak_activity = activity[peaks]


    # Sorting 

    order = np.argsort(
        peak_freqs
    )

    peaks = peaks[order]

    peak_freqs = peak_freqs[order]

    peak_amps = peak_amps[order]

    peak_activity = peak_activity[order]


    peak_to_group = {}



    clusters = bcps.cluster_peaks_by_frequency(
        peaks=np.arange(len(peaks)),
        frequency=peak_freqs,
        amplitude=peak_amps,
        peak_to_group=peak_to_group,
        cluster_gap_threshold=1.0e5,
        amp_ratio_threshold=0.6,
        valley_threshold=0.4
    )


    # Originalindices

    clusters_original = [
        peaks[c]
        for c in clusters
    ]


    print()
    print("----------------------------")
    print("No grouping pipeline")
    print("----------------------------")

    print(
        "Detected peaks:",
        len(peaks)
    )

    print(
        "Detected clusters:",
        len(clusters_original)
    )


    print(
        "Peaks in clusters:",
        sum(
            len(c)
            for c in clusters_original
        )
    )


    return (
        peaks,
        peak_freqs,
        peak_amps,
        peak_activity,
        clusters_original,
        resonance_mask,
        activity
    )

In [ ]:
peaks, resonance_mask, activity = pfat.find_peaks_adaptive(
        frequency,
        amplitude,
        prominence_resonance=0.003,
        distance_resonance=10,
        threshold_background=0.001,
        merge_background_points=7,
        activity_window=300,
        activity_fraction=0.40
    )


print("Detected peaks:", len(peaks))


(
    peaks,
    peak_freqs,
    peak_amps,
    peak_activity
) = bcps.build_peak_space(
    frequency,
    amplitude,
    peaks,
    activity
)


peak_to_group = {}



clusters = bcps.cluster_peaks_by_frequency(
    peaks=np.arange(len(peak_freqs)),
    frequency=peak_freqs,
    amplitude=peak_amps,
    peak_to_group=peak_to_group,
    cluster_gap_threshold=0.5e5,
    amp_ratio_threshold=0.6,
    valley_threshold=0.4
)



# zurück zu Originalindizes
clusters_original = [
    peaks[c]
    for c in clusters
]


print()
print("================================")
print("Cluster result")
print("================================")
print("Number of clusters:", len(clusters_original))
print(
    "Cluster sizes:",
    [len(c) for c in clusters_original]
)

In [ ]:
print("Number of Peaks:", len(peaks))
print("Number of Clusters:", len(clusters))

print(
    "Peaks in Clusters:",
    sum(len(c) for c in clusters)
)

In [ ]:
raw_peak_results = bcps.extract_peak_segments_raw(
    frequency,
    amplitude,
    clusters_original,
    peak_to_group
)

In [ ]:
# Shape Spectrum
spec_orig = normspec.build_shape_spectrum_from_clusters(
    raw_peak_results,
    frequency
)
# not normalised spectrum
spec_raw = normspec.build_raw_spectrum_from_clusters(
    raw_peak_results,
    frequency
)

In [ ]:
# Debug

print("raw peaks max:",
      max(np.max(r["y"]) for r in raw_peak_results))

print("spec_raw max:",
      np.max(spec_raw))

print("spec_orig max:",
      np.max(spec_orig))

In [ ]:
def plot_cluster_shapes(
    raw_peak_results,
    cluster_positions,
    sim_matrix,
    cluster_ids,
    similarity_threshold=0.9,
    figsize=(12, 8)
):

    shapes = {}

    counts = {}


    for cid in cluster_ids:

        shape = ros.build_cluster_mean_shape(
            cid,
            raw_peak_results,
            target_len=200
        )


        if shape is None:
            continue


        shapes[cid] = shape

        counts[cid] = sum(
            1
            for r in raw_peak_results
            if r["cluster_id"] == cid
        )


    valid_cluster_ids = [
        cid
        for cid in cluster_ids
        if cid in shapes
    ]


    shape_groups = ros.group_similar_shapes(
        valid_cluster_ids,
        sim_matrix,
        similarity_threshold=similarity_threshold
    )


    print("=" * 70)
    print("SHAPE GROUPS")
    print("=" * 70)

    for i, group in enumerate(shape_groups):

        total_peaks = sum(
            counts[cid]
            for cid in group
        )

        print(
            f"Shape {i+1}: "
            f"Clusters = {group} | "
            f"N clusters = {len(group)} | "
            f"N peaks = {total_peaks}"
        )


    print()


    fig, ax = ros.plt.subplots(
        figsize=figsize
    )


    x = np.linspace(
        0,
        1,
        200
    )

    cmap = ros.plt.cm.tab20


    for group_index, group in enumerate(shape_groups):

        group_shape = ros.build_group_mean_shape(
            group,
            shapes
        )


        if group_shape is None:
            continue


        color = cmap(
            group_index % 20
        )


        positions = [
            cluster_positions[cid] / 1e6
            for cid in group
            if cid in cluster_positions
        ]


        total_peaks = sum(
            counts[cid]
            for cid in group
        )


        cluster_text = ", ".join(
            str(cid)
            for cid in group
        )


        label = (
            f"Shape {group_index+1}: "
            #f"C[{cluster_text}] "
            f"(N={total_peaks})"
        )

        # mean shape

        ax.plot(
            x,
            group_shape,
            color=color,
            linewidth=2.5,
            alpha=0.9,
            label=label
        )

        # every peak shown in the background 

        for cid in group:

            ax.plot(
                x,
                shapes[cid],
                color=color,
                alpha=0.12,
                linewidth=0.8
            )



    ax.set_xlabel(
        "Normalized peak position"
    )

    ax.set_ylabel(
        "L2-normalized amplitude"
    )

    ax.set_title(
        "Mean normalized peak shapes grouped by similarity"
    )


    ax.grid(
        alpha=0.2
    )


    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8
    )


    ros.plt.tight_layout()

    ros.plt.show()

    return shape_groups

In [ ]:
templates = crosstemp.compute_cluster_templates(
    raw_peak_results
)


cluster_ids, sim_matrix = crosstemp.compute_similarity_matrix(
    templates
)

cluster_properties = crosstemp.get_cluster_properties(
    raw_peak_results,
    frequency,
    activity=activity,
    resonance_mask=resonance_mask
)

cluster_positions = {
    cid:data["position"]
    for cid,data in cluster_properties.items()
}

shape_groups = plot_cluster_shapes(
    raw_peak_results,
    cluster_positions,
    sim_matrix,
    cluster_ids,
    similarity_threshold=0.99
)

In [ ]:
clusters_new, removed_shape_clusters, remaining_cluster_ids = \
    ros.remove_overrepresented_shape_clusters(
        clusters,
        shape_groups,
        expected_repeats=5,
        max_factor=1.8
    )

In [ ]:
# Filter raw_peak_results according to clusters_new

raw_peak_results_new = [
    r
    for r in raw_peak_results
    if r["cluster_id"] in remaining_cluster_ids
]


print("=" * 70)
print("FILTERED RAW PEAK RESULTS")
print("=" * 70)

print(
    "Original raw peaks:",
    len(raw_peak_results)
)

print(
    "Remaining raw peaks:",
    len(raw_peak_results_new)
)

print(
    "Remaining cluster IDs:",
    sorted(remaining_cluster_ids)
)

In [ ]:
spec_new = normspec.build_shape_spectrum_from_clusters(
    raw_peak_results_new,
    frequency
)

In [ ]:

# Build templates from remaining peaks

templates = crosstemp.compute_cluster_templates(
    raw_peak_results_new
)


cluster_ids, sim_matrix = crosstemp.compute_similarity_matrix(
    templates
)


cluster_properties = crosstemp.get_cluster_properties(
    raw_peak_results_new,
    frequency,
    activity=activity,
    resonance_mask=resonance_mask
)

# Cluster properties

cluster_positions = {
    cid: data["position"]
    for cid, data in cluster_properties.items()
}


cluster_amplitudes = {
    cid: data["amplitude"]
    for cid, data in cluster_properties.items()
}


cluster_activity = {
    cid: data["activity"]
    for cid, data in cluster_properties.items()
}


valid_cluster_ids = [
    cid
    for cid in cluster_ids
    if cid in cluster_positions
    and cid in cluster_amplitudes
    and cid in cluster_activity
]


print("=" * 70)
print("GROUPING AFTER SHAPE REMOVAL")
print("=" * 70)

print(
    "Remaining clusters:",
    valid_cluster_ids
)

# PERIODIC GROUPING

groups = grouping.group_clusters_strict_periodic(
    sim_matrix,
    valid_cluster_ids,
    cluster_positions,
    cluster_amplitudes,
    cluster_activity,
    freq_min=params["freq_min"],
    freq_max=params["freq_max"],
    sim_threshold=0.6,
    min_repeats=params["min_repeats"],
    max_repeats=params["max_repeats"],
    harmonic_tolerance=5e4,
    max_spacing_error=5e4,
    max_harmonic_deviation=0.05
)

# SCORE GROUPS

groups = grouping.score_periodic_groups(
    groups,
    cluster_amplitudes,
    cluster_activity,
    valid_cluster_ids,
    sim_matrix
)

# PRINT RESULTS

print("\nDetected periodic groups\n")


for i, g in enumerate(groups):

    print(f"Group {i+1}")

    print(
        f"clusters = {g['clusters']}"
    )

    print(
        f"harmonics = {g['harmonics']}"
    )

    print(
        f"f_rev = {g['harmonic_spacing']/1e6:.6f} MHz"
    )

    print(
        f"fit error = {g['fit_error']/1e3:.2f} kHz"
    )

    print(
        f"score = {g['score']:.3f}"
    )

    print()

In [ ]:
for i, g in enumerate(groups):

    print(f"\n{'='*60}")
    print(f"Group {i+1}")
    print(f"{'='*60}")

    print(
        f"Recovered f_rev : {g['harmonic_spacing']/1e6:.6f} MHz"
    )

    print(
        f"Fit error        : {g['fit_error']/1e3:.2f} kHz"
    )

    print(
        f"Harmonics        : {g['harmonics']}"
    )

    print("\nClusters:")

    for cid, h in zip(g["clusters"], g["harmonics"]):

        print(
            f"  H={h:2d} | "
            f"Cluster {cid:2d} | "
            f"{cluster_positions[cid]/1e6:.6f} MHz | "
            f"Amp = {cluster_amplitudes[cid]:.3f}"
        )

In [ ]:
import copy
import itertools
import numpy as np

In [ ]:
groups_final = grouping.select_final_groups(
    groups
)

In [ ]:
periodicity_results = fh.analyze_group_periodicity(
    groups_final,
    cluster_positions
)

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.signal import find_peaks


In [ ]:
peak_families = fh.build_peak_families_robust(
    groups_final,
    raw_peak_results_new
)

In [ ]:
display(
    peak_families.sort_values(
        [
            "group_id",
            "family_id",
            "f_peak"
        ]
    )
    [
        [
            "family_id",
            "group_id",
            "cluster_id",
            "peak_index",
            "f_peak",
            "harmonic",
            "fundamental",
            "folded_frequency"
        ]
    ]
)

In [ ]:
fh.plot_mean_peak_families_styled(
    peak_families,
    frequency,
    spec_orig,
    n_points=4000,
    distance=10,             
    prominence_factor=0.03 
)

In [ ]:
print("\n====================")
print("SPEC_ORIG")
print("====================")

x_cont_orig, spec_cont_orig = csa.build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_orig,
    n_points=600000
)

In [ ]:
x_cont_x, spec_cont_x = csa.build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_orig,
    n_points=60000,
    baseline_floor=1e-12
)

np.savez(
    "continuous_spectrum_x.npz",
    frequency=x_cont_x,
    spectrum=spec_cont_x
)

In [ ]:
print("\n====================")
print("SPEC_RAW")
print("====================")

x_cont_raw_x, spec_cont_raw_x = csa.build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_raw,
    n_points=600000
)

In [ ]:
x_cont_raw_x, spectrum_cont_raw_x = csa.build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_raw,
    n_points = 600000
)

np.savez(
    "continuous_spectrum_raw_x.npz",
    frequency=x_cont_raw_x,
    spectrum=spectrum_cont_raw_x
)

In [ ]:
# nearest neightbor analysis 
df = peak_families.copy()

fundamentals = df.sort_values("f_peak").reset_index(drop=True)

freqs = fundamentals["f_peak"].values

print("\n===== PEAK POSITIONS (SORTED) =====")

display(
    fundamentals[
        [
            "family_id",
            "cluster_id",
            "f_peak",
            "harmonic",
            "fundamental"
        ]
    ]
)

#  NEAREST NEIGHBORS

diff_rows = []

for i in range(len(fundamentals) - 1):

    j = i + 1

    f1 = freqs[i]
    f2 = freqs[j]

    delta_f = f2 - f1

    diff_rows.append({
        "family_1": fundamentals["family_id"].iloc[i],
        "family_2": fundamentals["family_id"].iloc[j],

        "cluster_1": fundamentals["cluster_id"].iloc[i],
        "cluster_2": fundamentals["cluster_id"].iloc[j],

        "f1_Hz": f1,
        "f2_Hz": f2,

        "delta_f_Hz": delta_f,
        "relative_df": delta_f / f1 if f1 != 0 else np.nan
    })

df_diff = pd.DataFrame(diff_rows)

df_diff = df_diff.sort_values("delta_f_Hz").reset_index(drop=True)

print("\n===== NEAREST NEIGHBOR SPACING =====")

display(df_diff)